# Positional Embeddings: Teaching Transformers About Order

Transformers process all tokens in parallel, which means they have **no inherent notion of sequence order**.
Without positional information, the sentence "the cat sat on the mat" would be indistinguishable from
"mat the on sat cat the" to a transformer.

Positional embeddings solve this by injecting position information into the model. Over the years,
several approaches have emerged, each with different trade-offs:

| Method | Type | Key Idea | Used In |
|--------|------|----------|--------|
| **Sinusoidal** | Absolute, fixed | Sine/cosine waves at different frequencies | Original Transformer |
| **Learned** | Absolute, trainable | Embeddings learned via backprop | GPT-2, BERT |
| **RoPE** | Relative, fixed | Rotates query/key vectors by position angle | LLaMA, Mistral |
| **ALiBi** | Relative, no embeddings | Linear bias on attention scores | BLOOM, MPT |

In this notebook we will build, visualize, and compare all four methods.

In [ ]:
%matplotlib inline

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from positional_embeddings import SinusoidalPE, LearnedPE, RoPEPE, ALiBiPE, PositionalEmbeddingAnalyzer
from sinusoidal import sinusoidal
from rope import rope
from alibi import alibi_bias

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

---
## 1. Sinusoidal Positional Encoding

The original transformer (Vaswani et al., 2017) used fixed sinusoidal functions:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right) \quad \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

Each dimension uses a different frequency, so nearby positions share similar encodings while distant
positions become increasingly distinct. The key insight is that relative position can be expressed
as a linear function of the encoding, which lets the model learn to attend by relative position.

In [ ]:
# Create a sinusoidal PE module
embed_dim = 64
seq_len = 50
sin_pe = SinusoidalPE(embed_dim=embed_dim, max_seq_len=5000)

# Forward pass: provide a dummy input to extract the positional encodings
x_dummy = torch.zeros(1, seq_len, embed_dim)
sin_encoding = sin_pe(x_dummy)  # returns pe[:seq_len, :]

print(f"Input shape:    {x_dummy.shape}")
print(f"Encoding shape: {sin_encoding.shape}")
print(f"Value range:    [{sin_encoding.min():.3f}, {sin_encoding.max():.3f}]")

In [ ]:
# Visualize the sinusoidal encoding as a heatmap
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(sin_encoding[0].detach().numpy(), cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
ax.set_xlabel('Embedding Dimension')
ax.set_ylabel('Position')
ax.set_title('Sinusoidal Positional Encoding Heatmap')
plt.colorbar(im, ax=ax, label='Value')
plt.tight_layout()
plt.show()

Notice the characteristic wave patterns: lower dimensions (left) oscillate rapidly, while higher
dimensions (right) change very slowly. This multi-scale representation lets the model distinguish
positions at different granularities.

In [ ]:
# Show wave patterns for specific dimensions
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
dims_to_show = [0, 10, 30, 62]
positions = np.arange(seq_len)
encoding_np = sin_encoding[0].detach().numpy()

for ax, d in zip(axes.flat, dims_to_show):
    ax.plot(positions, encoding_np[:, d], linewidth=2, color='steelblue')
    ax.set_ylabel('Value')
    ax.set_title(f'Dimension {d} ({"sin" if d % 2 == 0 else "cos"})')
    ax.set_ylim(-1.2, 1.2)
    ax.grid(True, alpha=0.3)

axes[1, 0].set_xlabel('Position')
axes[1, 1].set_xlabel('Position')
plt.suptitle('Sinusoidal Waves at Different Dimensions', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Use the low-level sinusoidal() function to build encodings position by position
manual_enc = torch.stack([sinusoidal(pos, embed_dim) for pos in range(10)])
print(f"Manual encoding shape: {manual_enc.shape}")
print(f"Position 0, first 8 dims: {manual_enc[0, :8].tolist()}")
print(f"Position 5, first 8 dims: {manual_enc[5, :8].tolist()}")

---
## 2. Learned Positional Embeddings

Instead of a fixed formula, learned PE simply assigns a trainable embedding vector to each position
via `nn.Embedding`. This is what GPT-2 and BERT use.

**Pros:** Maximum flexibility -- the model can learn whatever positional patterns help the task.

**Cons:** Cannot generalize to positions longer than the training set. Adds parameters.

In [ ]:
# Create a learned PE module
learned_pe = LearnedPE(embed_dim=embed_dim, max_seq_len=5000)

x_dummy = torch.zeros(1, seq_len, embed_dim)
learned_encoding = learned_pe(x_dummy)

print(f"Encoding shape: {learned_encoding.shape}")
print(f"Value range (random init): [{learned_encoding.min():.3f}, {learned_encoding.max():.3f}]")
print(f"\nLearnable parameters: {sum(p.numel() for p in learned_pe.parameters()):,}")

In [ ]:
# Visualize the randomly initialized learned embeddings
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Learned (random init)
im0 = axes[0].imshow(learned_encoding[0].detach().numpy(), cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
axes[0].set_title('Learned PE (Random Initialization)')
axes[0].set_xlabel('Embedding Dimension')
axes[0].set_ylabel('Position')
plt.colorbar(im0, ax=axes[0])

# Sinusoidal for comparison
sin_enc_compare = sin_pe(x_dummy)
im1 = axes[1].imshow(sin_enc_compare[0].detach().numpy(), cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
axes[1].set_title('Sinusoidal PE (Fixed)')
axes[1].set_xlabel('Embedding Dimension')
axes[1].set_ylabel('Position')
plt.colorbar(im1, ax=axes[1])

plt.suptitle('Learned vs. Sinusoidal: Before Training', fontsize=13)
plt.tight_layout()
plt.show()

The learned embeddings start as random noise. After training, they typically converge to smooth
patterns that resemble (but are not identical to) sinusoidal encodings. The key disadvantage is
that positions beyond `max_seq_len` are simply undefined.

---
## 3. Rotary Position Embeddings (RoPE)

RoPE (Su et al., 2021) encodes position by **rotating** query and key vectors in 2D subspaces.
For each pair of dimensions $(2i, 2i+1)$, the embedding applies a rotation by angle $\theta_i \cdot pos$:

$$\begin{pmatrix} q'_{2i} \\ q'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}$$

The dot product between rotated queries and keys naturally depends on their **relative** position,
making RoPE a relative position encoding that works elegantly with the attention mechanism.

In [ ]:
# Create RoPE and apply to random input
rope_pe = RoPEPE(embed_dim=embed_dim, max_seq_len=5000)

x_input = torch.randn(1, seq_len, embed_dim)
x_rotated = rope_pe(x_input)

print(f"Input shape:   {x_input.shape}")
print(f"Output shape:  {x_rotated.shape}")
print(f"\nInput norm (pos 0):    {x_input[0, 0].norm():.4f}")
print(f"Rotated norm (pos 0):  {x_rotated[0, 0].norm():.4f}")
print("\nRoPE preserves vector norms (rotation is norm-preserving).")

In [ ]:
# Visualize the rotation effect
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Before rotation
im0 = axes[0].imshow(x_input[0].detach().numpy()[:20], cmap='RdBu', aspect='auto')
axes[0].set_title('Before RoPE')
axes[0].set_xlabel('Dimension')
axes[0].set_ylabel('Position')
plt.colorbar(im0, ax=axes[0])

# After rotation
im1 = axes[1].imshow(x_rotated[0].detach().numpy()[:20], cmap='RdBu', aspect='auto')
axes[1].set_title('After RoPE')
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Position')
plt.colorbar(im1, ax=axes[1])

# Difference
diff = (x_rotated[0] - x_input[0]).detach().numpy()[:20]
im2 = axes[2].imshow(diff, cmap='RdBu', aspect='auto')
axes[2].set_title('Difference (After - Before)')
axes[2].set_xlabel('Dimension')
axes[2].set_ylabel('Position')
plt.colorbar(im2, ax=axes[2])

plt.suptitle('RoPE: Rotation Effect on Token Embeddings', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Use the low-level rope() function directly
x_small = torch.randn(5, 16)  # 5 tokens, 16 dims
x_rotated_manual = rope(x_small)

print(f"Input shape:  {x_small.shape}")
print(f"Output shape: {x_rotated_manual.shape}")

# Verify norm preservation
norms_before = x_small.norm(dim=-1)
norms_after = x_rotated_manual.norm(dim=-1)
print(f"\nNorm changes: {(norms_after - norms_before).abs().max():.6f} (should be ~0)")

---
## 4. Attention with Linear Biases (ALiBi)

ALiBi (Press et al., 2022) takes a radically different approach: instead of modifying the token
embeddings, it adds a **linear bias** directly to the attention scores:

$$\text{attention}(q_i, k_j) = q_i^T k_j - m \cdot |i - j|$$

where $m$ is a head-specific slope. Each head gets a different slope (geometric series), so some
heads focus locally while others attend more globally. This requires **zero additional parameters**
and generalizes well to longer sequences than seen during training.

In [ ]:
# Create ALiBi for 8 attention heads
num_heads = 8
alibi_pe = ALiBiPE(num_heads=num_heads)

alibi_seq_len = 20
alibi_bias_tensor = alibi_pe(alibi_seq_len, device=torch.device('cpu'))

print(f"Bias shape: {alibi_bias_tensor.shape}  (num_heads, seq_len, seq_len)")
print(f"\nHead slopes: {alibi_pe.slopes.tolist()}")
print(f"\nHead 0 bias for query=5: {alibi_bias_tensor[0, 5, :8].tolist()}")

In [ ]:
# Visualize per-head attention biases
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for h in range(num_heads):
    ax = axes[h // 4, h % 4]
    im = ax.imshow(alibi_bias_tensor[h].detach().numpy(), cmap='RdBu', aspect='auto')
    ax.set_title(f'Head {h} (slope={alibi_pe.slopes[h]:.4f})')
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('ALiBi Attention Biases Per Head', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Show the linear decay pattern: bias for a fixed query position across all key positions
query_pos = 15
fig, ax = plt.subplots(figsize=(10, 5))

for h in range(num_heads):
    bias_row = alibi_bias_tensor[h, query_pos, :].detach().numpy()
    ax.plot(range(alibi_seq_len), bias_row, label=f'Head {h}', marker='o', markersize=3)

ax.axvline(x=query_pos, color='black', linestyle='--', alpha=0.5, label=f'Query pos={query_pos}')
ax.set_xlabel('Key Position')
ax.set_ylabel('Bias Value')
ax.set_title(f'ALiBi Linear Decay from Query Position {query_pos}')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Use the low-level alibi_bias() function
bias_manual = alibi_bias(seq_len=10, heads=4)
print(f"Manual bias shape: {bias_manual.shape}")
print(f"Head 0, row 3: {bias_manual[0, 3].tolist()}")

The steep slopes (Head 0) create a strong recency bias -- the model pays attention almost exclusively
to nearby tokens. The gentle slopes (Head 7) allow much broader attention. This diversity of attention
patterns is built-in, requiring no learning.

---
## 5. Side-by-Side Comparison

Now let us use the `PositionalEmbeddingAnalyzer` to compare all four methods systematically.

In [ ]:
# Create the analyzer
analyzer = PositionalEmbeddingAnalyzer(embed_dim=64, seq_len=20)

# Retrieve all encodings
sin_enc, learned_enc, rope_enc, alibi_enc = analyzer.get_encodings()

print(f"Sinusoidal encoding: {sin_enc.shape}")
print(f"Learned encoding:    {learned_enc.shape}")
print(f"RoPE encoding:       {rope_enc.shape}")
print(f"ALiBi encoding:      {alibi_enc.shape}  (heads x seq_len x seq_len)")

In [ ]:
# 2x2 heatmap comparison of all four methods
analyzer.visualize_all_methods()

**Observations:**
- **Sinusoidal** shows clean, structured wave patterns with decreasing frequency across dimensions.
- **Learned** appears as random noise since the model has not been trained yet.
- **RoPE** shows the rotation frequencies (similar structure to sinusoidal, but applied multiplicatively).
- **ALiBi** is fundamentally different: it is a position-position bias matrix, not a position-dimension encoding.

---
## 6. Embedding Norm Evolution

How does the L2 norm of the positional encoding change as we move to later positions?
Ideally, all positions should have similar norms to avoid scaling issues.

In [ ]:
# Norm vs position for sinusoidal, learned, and RoPE
analyzer.plot_position_norm_evolution()

**Analysis:**
- **Sinusoidal** has perfectly constant norm across all positions (by construction -- it is a sum of sin/cos waves).
- **Learned** norms vary randomly since the embeddings have not been trained.
- **RoPE** norms depend on the input vectors being rotated, so they vary with the random input.

Constant norms are desirable because they prevent position-dependent scaling of the residual stream.

---
## 7. Position Distance Matrices

A good positional encoding should make nearby positions more similar than distant ones.
We measure this with cosine similarity between all pairs of position encodings.

In [ ]:
# Cosine similarity matrices for sinusoidal, learned, and RoPE
analyzer.plot_position_distance_matrix()

**Analysis:**
- **Sinusoidal** shows a beautiful banded structure: positions close together have high similarity
  (warm colors on the diagonal), with similarity decreasing smoothly with distance.
- **Learned** (untrained) shows no meaningful structure -- it is essentially random.
- **RoPE** similarity depends on the input content, but the rotational structure ensures that
  relative position information is encoded in the dot products.

---
## 8. Ablation: Embedding Dimension Impact

How does the embedding dimension affect the ability of positional encodings to differentiate
between positions? We measure "position differentiation" as the average pairwise distance
between position encodings.

In [ ]:
# Ablation over dimensions [8, 16, 32, 64, 128]
analyzer.ablate_embedding_dim()

**Interpretation:** Higher embedding dimensions generally lead to greater position differentiation.
This makes intuitive sense: more dimensions provide more "room" for positions to be distinct.
For sinusoidal encodings, the increase is smooth and predictable. Learned embeddings show more
variance due to random initialization.

---
## 9. Position Extrapolation: Beyond Training Length

A critical practical question: what happens when a model encounters sequences longer than those
seen during training? This is where the methods diverge dramatically.

In [ ]:
# Train length vs. extrapolation length
train_len = 50
extrap_len = 200
dim = 64

# Sinusoidal: can extrapolate naturally (deterministic formula)
sin_pe_extrap = SinusoidalPE(dim, max_seq_len=extrap_len)
x_extrap = torch.zeros(1, extrap_len, dim)
sin_enc_extrap = sin_pe_extrap(x_extrap)[0].detach()

# Learned: limited to max_seq_len
learned_pe_extrap = LearnedPE(dim, max_seq_len=train_len)
x_train = torch.zeros(1, train_len, dim)
learned_enc_train = learned_pe_extrap(x_train)[0].detach()

# ALiBi: extrapolates by design
alibi_pe_extrap = ALiBiPE(num_heads=8)
alibi_train = alibi_pe_extrap(train_len, device=torch.device('cpu'))
alibi_extrap = alibi_pe_extrap(extrap_len, device=torch.device('cpu'))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Sinusoidal: smooth extrapolation
sin_norms = sin_enc_extrap.norm(dim=1).numpy()
axes[0].plot(range(extrap_len), sin_norms, color='steelblue')
axes[0].axvline(x=train_len, color='red', linestyle='--', label=f'Train boundary ({train_len})')
axes[0].set_title('Sinusoidal: Norm vs Position')
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Norm')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Learned: hard boundary
learned_norms = learned_enc_train.norm(dim=1).numpy()
axes[1].plot(range(train_len), learned_norms, color='darkorange')
axes[1].axvline(x=train_len, color='red', linestyle='--', label=f'Max position ({train_len})')
axes[1].set_title('Learned: Cannot Extrapolate')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Norm')
axes[1].set_xlim(0, extrap_len)
axes[1].annotate('Undefined!', xy=(train_len + 20, learned_norms.mean()),
                 fontsize=14, color='red', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# ALiBi: linear bias scales naturally
alibi_diag_train = torch.diagonal(alibi_train[0], offset=-1).numpy()
alibi_diag_extrap = torch.diagonal(alibi_extrap[0], offset=-1).numpy()
axes[2].plot(range(len(alibi_diag_extrap)), alibi_diag_extrap, color='forestgreen',
             label='Adjacent-token bias')
axes[2].axvline(x=train_len, color='red', linestyle='--', label=f'Train boundary ({train_len})')
axes[2].set_title('ALiBi: Smooth Extrapolation')
axes[2].set_xlabel('Position')
axes[2].set_ylabel('Bias Value')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Position Extrapolation Beyond Training Length', fontsize=14)
plt.tight_layout()
plt.show()

**Extrapolation summary:**
- **Sinusoidal:** Extrapolates perfectly (deterministic formula), but the model may not have learned
  to use these unseen position patterns.
- **Learned:** Hard failure -- positions beyond `max_seq_len` are simply undefined.
- **RoPE:** Extrapolates the rotation angles, but quality degrades. Techniques like NTK-aware scaling
  and YaRN help extend context.
- **ALiBi:** Best extrapolation -- the linear bias pattern is position-independent and extends naturally.

---
## 10. Parameter Count Comparison

Different methods have very different memory footprints.

In [ ]:
# Compare parameter counts
dim = 64
max_len = 5000
n_heads = 8

sin_model = SinusoidalPE(dim, max_len)
learned_model = LearnedPE(dim, max_len)
rope_model = RoPEPE(dim, max_len)
alibi_model = ALiBiPE(n_heads)

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    buffers = sum(b.numel() for b in model.buffers())
    return total, buffers

methods = ['Sinusoidal', 'Learned', 'RoPE', 'ALiBi']
models = [sin_model, learned_model, rope_model, alibi_model]

print(f"{'Method':<15} {'Trainable Params':>18} {'Buffer Elements':>18} {'Total Memory':>15}")
print('-' * 70)
for name, model in zip(methods, models):
    params, buffers = count_params(model)
    total = params + buffers
    print(f"{name:<15} {params:>18,} {buffers:>18,} {total:>15,}")

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
param_counts = [count_params(m)[0] for m in models]
buffer_counts = [count_params(m)[1] for m in models]

x_pos = np.arange(len(methods))
bars1 = ax.bar(x_pos, param_counts, width=0.4, label='Trainable Parameters', color='steelblue')
bars2 = ax.bar(x_pos + 0.4, buffer_counts, width=0.4, label='Buffers (Fixed)', color='coral')

ax.set_xlabel('Method')
ax.set_ylabel('Count')
ax.set_title(f'Parameter Counts (embed_dim={dim}, max_len={max_len})')
ax.set_xticks(x_pos + 0.2)
ax.set_xticklabels(methods)
ax.legend()
ax.set_yscale('symlog')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

**Parameter efficiency:**
- **Sinusoidal:** Zero trainable parameters. The encodings are stored as fixed buffers.
- **Learned:** `max_seq_len x embed_dim` trainable parameters (320,000 for our settings). This scales
  linearly with both sequence length and embedding dimension.
- **RoPE:** Zero trainable parameters. Only stores the inverse frequency vector as a buffer.
- **ALiBi:** Zero trainable parameters. Only stores the head slopes (one scalar per head).

---
## 11. Cosine Similarity at Different Relative Distances

Let us directly measure how cosine similarity between position encodings decays
with relative distance. A smooth, monotonic decay is generally desirable.

In [ ]:
# Cosine similarity vs. relative distance
from sklearn.metrics.pairwise import cosine_similarity

analysis_len = 50
analysis_dim = 64

sin_pe_analysis = SinusoidalPE(analysis_dim)
x_analysis = torch.zeros(1, analysis_len, analysis_dim)
sin_enc_analysis = sin_pe_analysis(x_analysis)[0].detach().numpy()

sim_matrix = cosine_similarity(sin_enc_analysis)

# Average similarity at each relative distance
max_dist = analysis_len - 1
avg_sim = []
for d in range(max_dist):
    sims = [sim_matrix[i, i + d] for i in range(analysis_len - d)]
    avg_sim.append(np.mean(sims))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(max_dist), avg_sim, color='steelblue', linewidth=2)
ax.set_xlabel('Relative Distance')
ax.set_ylabel('Average Cosine Similarity')
ax.set_title('Sinusoidal PE: Similarity Decay with Distance')
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

The sinusoidal encoding shows a smooth decay in similarity as relative distance increases,
with some oscillation due to the periodic nature of the sine/cosine functions. This oscillation
means that very distant positions can occasionally have moderately high similarity -- a known
limitation of fixed sinusoidal encodings.

---
## Key Insights

### When to use which method?

**Sinusoidal PE:**
- Good baseline with zero trainable parameters.
- Mathematically elegant with built-in relative position information.
- Largely superseded by RoPE and ALiBi in modern architectures.

**Learned PE:**
- Maximum flexibility; the model learns whatever works best.
- Cannot extrapolate beyond training sequence length -- a hard limitation.
- Adds `max_seq_len x embed_dim` parameters, which can be significant.
- Still used in BERT-style encoder models with fixed context windows.

**RoPE (Rotary Position Embeddings):**
- The dominant choice for modern LLMs (LLaMA, Mistral, Qwen).
- Naturally encodes relative position through rotation.
- Preserves vector norms -- important for training stability.
- Can be extended to longer contexts via NTK-aware scaling or YaRN.
- Zero additional parameters.

**ALiBi (Attention with Linear Biases):**
- Best length extrapolation out of the box -- no modifications needed.
- Zero parameters; biases are computed on the fly.
- Different heads automatically specialize (local vs. global attention).
- Used in BLOOM, MPT, and other models prioritizing long-context generalization.

### Summary Table

| Property | Sinusoidal | Learned | RoPE | ALiBi |
|----------|-----------|---------|------|-------|
| Trainable params | None | O(L x d) | None | None |
| Position type | Absolute | Absolute | Relative | Relative |
| Extrapolation | Limited | None | With tricks | Excellent |
| Modern usage | Rare | Encoder models | Most LLMs | Some LLMs |